# 11 — Difference-in-Differences (DiD)

Exploit the pre/post AI rollout timeline to estimate causal effects via DiD.

**Setup**
- Pre-period: 2023 — all agents human-only
- Post-period: 2024 — AI assistance rolled out to ~50 % of interactions
- Parallel trends assumption: in absence of treatment, treated and control groups would have followed the same trend

**Pipeline**
1. Simple DiD (group means)
2. Regression DiD with covariates (HC1 standard errors)
3. Parallel trends visual check
4. Placebo period test

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.generate import generate_synthetic_conversations
from src.analysis.did import difference_in_differences, regression_did
from src.sensitivity.robustness import placebo_test

sns.set_theme(style='whitegrid')
%matplotlib inline

COVARIATES = ['issue_severity', 'customer_tenure', 'time_of_day', 'agent_experience']
OUTCOMES   = ['resolution_time', 'satisfaction_score', 'escalated']
TREATMENT  = 'ai_assisted'
PERIOD_COL = 'created_at'
CUTOFF     = pd.Timestamp('2024-01-01')

## 1. Load data

In [ ]:
CSV = '../data/synthetic_conversations.csv'
if os.path.exists(CSV):
    df = pd.read_csv(CSV, parse_dates=['created_at'])
else:
    df = generate_synthetic_conversations(n=2000, seed=42)

df['post'] = (df['created_at'] >= CUTOFF).astype(int)
print(df.groupby(['post', TREATMENT]).size().rename(index={0: 'pre/human', 1: 'post/AI'}))

## 2. Parallel trends visual check

In [ ]:
# Weekly means by treatment group
weekly = (
    df.set_index('created_at')
    .groupby([pd.Grouper(freq='ME'), TREATMENT])
    [OUTCOMES]
    .mean()
    .reset_index()
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
for ax, outcome in zip(axes, OUTCOMES):
    for treat, grp in weekly.groupby(TREATMENT):
        label = 'AI assisted' if treat else 'Human only'
        ax.plot(grp['created_at'], grp[outcome], marker='o', markersize=3, label=label)
    ax.axvline(CUTOFF, color='red', linestyle='--', linewidth=1.5, label='AI rollout')
    ax.set_title(outcome)
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=30)
plt.suptitle('Monthly mean outcomes by treatment group (parallel trends check)', y=1.02)
plt.tight_layout()
plt.show()

## 3. Simple DiD — group means

In [ ]:
for outcome in OUTCOMES:
    did_est, cells = difference_in_differences(
        df,
        time_col=PERIOD_COL,
        period_cutoff=CUTOFF,
        treatment_col=TREATMENT,
        outcome_col=outcome,
    )
    print(f'\n{outcome}')
    print(f"  Pre:  treated={cells['treated_pre']:.3f}  control={cells['control_pre']:.3f}")
    print(f"  Post: treated={cells['treated_post']:.3f}  control={cells['control_post']:.3f}")
    print(f"  DiD estimate = {did_est:+.3f}")

## 4. Regression DiD with covariates

In [ ]:
reg_results = []
for outcome in OUTCOMES:
    model, coef = regression_did(
        df,
        time_col=PERIOD_COL,
        period_cutoff=CUTOFF,
        treatment_col=TREATMENT,
        outcome_col=outcome,
        covariates=COVARIATES,
    )
    p = model.pvalues.get(f'{TREATMENT}:period', np.nan)
    ci = model.conf_int().loc[f'{TREATMENT}:period'] if f'{TREATMENT}:period' in model.conf_int().index else (np.nan, np.nan)
    reg_results.append({
        'outcome': outcome,
        'DiD_coef': coef,
        'p_value': p,
        'CI_low': ci[0],
        'CI_high': ci[1],
    })

pd.DataFrame(reg_results).set_index('outcome').round(4)

In [ ]:
# Full regression summary for resolution_time
model_rt, _ = regression_did(
    df, PERIOD_COL, CUTOFF, TREATMENT, 'resolution_time', COVARIATES
)
print(model_rt.summary())

## 5. Coefficient plot

In [ ]:
rdf = pd.DataFrame(reg_results).set_index('outcome')

fig, ax = plt.subplots(figsize=(7, 4))
y = range(len(rdf))
ax.scatter(rdf['DiD_coef'], y, zorder=3, s=60)
for i, (outcome, row) in enumerate(rdf.iterrows()):
    ax.plot([row['CI_low'], row['CI_high']], [i, i], linewidth=2)
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_yticks(list(y))
ax.set_yticklabels(rdf.index)
ax.set_xlabel('DiD coefficient (95 % CI)')
ax.set_title('Regression DiD estimates with covariates')
plt.tight_layout()
plt.show()

## 6. Placebo period test

Falsification: assign a fake rollout date in the **pre-period**. If parallel trends holds and there is no actual pre-trend, the DiD estimate on the placebo split should be near zero.

In [ ]:
# Use only pre-period data; split it in half as a fake rollout
pre_df = df[df['post'] == 0].copy()
fake_cutoff = pd.Timestamp('2023-07-01')

print('Placebo DiD on pre-period only (fake rollout = 2023-07-01):')
for outcome in OUTCOMES:
    placebo_model, placebo_coef = regression_did(
        pre_df, PERIOD_COL, fake_cutoff, TREATMENT, outcome, COVARIATES
    )
    p = placebo_model.pvalues.get(f'{TREATMENT}:period', np.nan)
    print(f'  {outcome:25s}  coef={placebo_coef:+.3f}  p={p:.3f}')

Placebo coefficients near zero with large p-values support the parallel trends assumption.